# Finance agent
Simple Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet
!pip install --force-reinstall -U -r requirements-dev.txt --quiet

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.


In [ ]:
%%writefile strands_claude.py
"""Simple strands agent demo"""

# pylint disable:W0718

import os
from typing import Optional

from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext


MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = os.getenv("AWS_REGION", "us-east-1")
SYSTEM_PROMPT = """
You are the Financial Assistant, a sophisticated financial specialist.
PURPOSE:
- Help users with finance related questions concisely 
- use calculator tool, if appropriate, to perform calculations

If queries are off-topic, remind user of your purpose
"""

# Initialize agent application wrapper
app = BedrockAgentCoreApp()


def initialize_agent(request_ctx: RequestContext):
    """Initialize the agent with  tools"""
    print(request_ctx)

    model = BedrockModel(
        model_id=MODEL_ID,
    )
    agent = Agent(
        tools=[calculator],
        model=model,
        system_prompt=SYSTEM_PROMPT
    )
    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context: Optional[RequestContext] = None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)

    agent = initialize_agent(context)
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "finance_agent__demo"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY'
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
# !agentcore launch \
#     --env BEDROCK_MODEL_ID=us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
#     --env AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true \
#     --env AWS_AGENTIC_INSTRUMENTATION=disabled \
#     --env AWS_AGENTIC_INSTRUMENTATION_OPT_IN=false \
#     --env AGENT_OBSERVABILITY_ENABLED=true

!agentcore launch \
    --env BEDROCK_MODEL_ID=us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
    --env AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true \
    --env OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_span_attributes_only


### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
!agentcore status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload


In [ ]:
!agentcore invoke '{"prompt": "what is ROTH 401K?"}'


In [ ]:
!agentcore invoke '{"prompt": "ny capital"}'


## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run

### Congratulations!